In [ ]:
import random
import json
import sys
import numpy as np

from typing import Dict, Any, Sequence, Optional 
from importnb import Notebook

with Notebook():
    # from LabTrajectory_RandomWalk import simulate_viewport_with_tiles
    from LabTrajectory_360Dataset import simulate_viewport_with_tiles, load_all_yaws_pitches
    from LabTrajectory_Pantelis import load_trajectories, simulate_viewport

sys.path.append(r'c:\Users\es25591\Workspace\CacheVideoPredict360\Sources')
# sys.path.append('/home/eduardo/Workspace/CacheVideoPredict360/Sources')
from Common.Utils import zipf, poisson_per_users

# filename_path_ = '/home/eduardo/Workspace/CacheVideoPredict360/Dataset/2'
# filename_path_ = r'c:\Users\es25591\Workspace\CacheVideoPredict360\Dataset\2'
# filename_path_ = '/home/eduardo/Workspace/CacheVideoPredict360/Dataset/Trajectories'
filename_path_ = r'c:\Users\es25591\Workspace\CacheVideoPredict360\Dataset\Trajectories'


In [ ]:
class UserTileRequestEvents:
    def __init__(
        self,
        n_nodes: int = 1,
        n_users: int = 1,
        step_size: float = 5.0,
        alpha: float = 1.0,
        n_videos: int = 100,
        n_gops: int = 60,
        n_layers: int = 1,
        n_tiles: int = 4,
        n: int = 4,
        m: int = 3,
        arrival_rate: float = 10.0,
        users_viewport_tiles: Optional[Sequence[Any]] = [],
        requested_videos: Optional[Sequence[Any]] = [],
        users_arrivals: Optional[Sequence[Any]] = []
    ):
        self.step_count = 0
        self.user_gop_counter = [0 for _ in range(n_users)]
        self.user_node_map = [random.randrange(n_nodes) for _ in range(n_users)]
        # self.user_node_map = [i % n_nodes for i in range(n_users)]

        self.n_nodes = n_nodes
        self.n_users = n_users
        self.users_done = 0

        self.step_size = step_size
        self.alpha = alpha
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_layers = n_layers
        self.n_tiles = n_tiles
        self.n = n
        self.m = m
        self.users_viewport_tiles = users_viewport_tiles
        self.requested_videos = requested_videos
        self.users_arrivals = users_arrivals
        self.arrival_rate = arrival_rate

    def gen_request_for_user(
        self, 
        u_id: int, 
        p_id: int,
        du_bitmaps,
        mec_bitmap
    ) -> Dict[str, Any]:
        video = self.requested_videos[u_id]
        gop = self.user_gop_counter[u_id]

        # Initial step, no request
        if gop == 0:
            self.user_gop_counter[u_id] += 1
            return {
                "gop": gop,
                "u": u_id,
                "p": p_id, 
                "layer": 0,
                "video": video,
                "viewport": None,
                "tiles": []
            }
        gop -= 1  # Adjust step for 0-indexing

        # --- NEW CACHE HIERARCHY LOGIC ---
        tiles = []
        for y in range(self.m):
            for x in range(self.n):
                is_in_du = du_bitmaps[p_id][video][0][y * self.n + x][gop] > 0 if du_bitmaps is not None else False
                is_in_mec = mec_bitmap[video][0][y * self.n + x][gop] > 0 if mec_bitmap is not None else False

                alpha_p_u = 0 # Served by DU (Edge)
                alpha_M_u = 0 # Served by MEC (Regional)
                alpha_C_u = 0 # Served by Cloud

                if is_in_du:
                    alpha_p_u = 1
                elif is_in_mec:
                    alpha_M_u = 1
                else:
                    alpha_C_u = 1

                tile = {
                    "tile": y * self.n + x,
                    "layer": 0,
                    "size": 2e6 / self.n_tiles,
                    "events": {
                        "alpha_p_u": alpha_p_u, 
                        "alpha_M_u": alpha_M_u, 
                        "alpha_C_u": alpha_C_u, 
                        "beta_p_u": 0, 
                        "beta_M_u": 0
                    }
                }
                tiles.append(tile)
        
        # 2. Request Enhancement Layer (Layer 1)
        current_viewport = self.users_viewport_tiles[u_id][gop]
        for x, y in current_viewport:
            if x < 0 or x >= self.n or y < 0 or y >= self.m:
                continue
            
            is_in_du = du_bitmaps[p_id][video][1][y * self.n + x][gop] > 0 if du_bitmaps is not None else False
            is_in_mec = mec_bitmap[video][1][y * self.n + x][gop] > 0 if mec_bitmap is not None else False
            alpha_p_u = 0
            alpha_M_u = 0
            alpha_C_u = 0

            if is_in_du:
                alpha_p_u = 1
            elif is_in_mec:
                alpha_M_u = 1
            else:
                alpha_C_u = 1

            tile = {
                "tile": y * self.n + x,
                "layer": 1,
                "size": 15e6 / self.n_tiles,
                "events": {
                    "alpha_p_u": alpha_p_u, 
                    "alpha_M_u": alpha_M_u, 
                    "alpha_C_u": alpha_C_u, 
                    "beta_p_u": 0, 
                    "beta_M_u": 0
                }
            }
            tiles.append(tile)

        self.user_gop_counter[u_id] += 1
        
        if self.user_gop_counter[u_id] >= self.n_gops + 1:
            self.users_done += 1

        current_viewport = np.array(
            [y * self.n + x for x, y in current_viewport], dtype=int
        )

        return {
            "gop": gop + 1,
            "u": u_id,
            "p": p_id, 
            "video": video,
            "viewport": current_viewport,
            "tiles": tiles,
        }

    def step(self, du_bitmaps, mec_bitmap):
        reqs = []
        for i in range(self.n_users):
            if (
                self.users_arrivals[i] <= self.step_count and 
                self.user_gop_counter[i] < self.n_gops + 1
            ):
                p = self.user_node_map[i]
                req = self.gen_request_for_user(i, p, du_bitmaps, mec_bitmap)
                reqs.append(req)

        self.step_count += 1
        return reqs
    
    def user_is_done(self, u_id: int) -> bool:
        return self.user_gop_counter[u_id] >= self.n_gops + 1

    def get_user_gop(self, u_id: int) -> int:
        return self.user_gop_counter[u_id]

    def reset(self, **kwargs):
        self.step_count = 0
        self.user_gop_counter = [0 for _ in range(self.n_users)]
        self.users_done = 0

        self.users_arrivals = poisson_per_users(
            total_users=self.n_users,
            rate_per_minute=self.arrival_rate
        )
        
        self.requested_videos = zipf(
            samples=self.n_users, 
            total_videos=self.n_videos, 
            alpha=self.alpha
        )
        self.requested_videos = [int(v-1) for v in self.requested_videos]

        ### Load dataset based on Pantelis' traces  ###
        trajectories = load_trajectories(
            filename_path_,
        )
        self.users_viewport_tiles = []
        for i in range(self.n_users):
            video = self.requested_videos[i]
            viewport_tiles = simulate_viewport(
                video, 
                trajectories
            )
            self.users_viewport_tiles.append(viewport_tiles)
        
        ### Load and shuffle viewport traces so users get randomized trajectories
        # yaws, pitches = load_all_yaws_pitches(
        #     filename_path_
        # )
        # viewport_data = list(zip(yaws, pitches))
        # random.shuffle(viewport_data)
        
        # self.users_viewport_tiles = []
        # for yaw, pitch in viewport_data:
        #     viewport_tiles, _ = simulate_viewport_with_tiles(
        #         num_steps=len(yaw),
        #         n=self.n,
        #         fov_yaw=100,
        #         fov_pitch=50,
        #         yaws=yaw,
        #         pitches=pitch,
        #     )
        #     self.users_viewport_tiles.append(viewport_tiles)
        
        ### Load dataset based on the LabTrajectory_RandomWalk ###
        # self.users_viewport_tiles = []
        # for _ in range(self.n_users):
        #     _, _, viewport_tiles, _ = simulate_viewport_with_tiles(
        #         num_steps=self.n_gops,
        #         n=self.n,
        #         fov_yaw=90,
        #         fov_pitch=50,
        #         damping=0.99,
        #         step_size=self.step_size,
        #         start_yaw=180,
        #         start_pitch=0
        #     )
        #     self.users_viewport_tiles.append(viewport_tiles)
        ### End of dataset loading ###

        info = {
            "viewport_tiles": self.users_viewport_tiles,
            "requested_videos": self.requested_videos,
            "users_arrivals": self.users_arrivals,
            "users_requests": []
        }
        
        return None, info

In [4]:
# Validation helpers
def validate_request_struct(req, num_tiles: int) -> bool:
    print(req)
    assert set([
        'gop','u','p','video','tiles'
    ]).issubset(req.keys()), 'Missing top-level keys'
    
    tiles = req['tiles']
    
    # Allow empty tiles for initial step (gop == 0), otherwise expect full base layer
    if len(tiles) > 0:
        assert len([
            t for t in tiles if t['layer'] == 0
        ]) == num_tiles, 'Base layer tiles count mismatch'
    
    for t in tiles:
        for k in ['tile','layer','size','events']:
            assert k in t, f'Missing tile key {k}'
        
        ev = t['events']
        
        for ek in ['alpha_p_u','alpha_M_u','alpha_C_u','beta_p_u','beta_M_u']:
            assert ek in ev, f'Missing event key {ek}'
    
    return True

In [5]:
if __name__ == '__main__':
    steps_to_run = 3
    n_users = 10
    step_size = 5.0
    alpha = 1.0
    n_videos = 100
    n_gops = 60
    n_layers = 2    # base + enhancement
    n = 4           # grid dimension (n x n)
    m = 3
    n_tiles = n * m # total tiles
    
    print('='*5, 'UserTileRequestEvents Test Harness', '='*5)
    print(f'Grid: {n}x{m} -> {n_tiles} tiles | Users: {n_users} | Layers: {n_layers}')
    print('Running steps...')

    user_env = UserTileRequestEvents(
        n=n,
        n_gops=n_gops,
        n_nodes=5,
        n_users=n_users,
        step_size=step_size,
        alpha=alpha,
        n_videos=n_videos,
        n_layers=n_layers,
        n_tiles=n_tiles
    )

    user_env.reset()
    user_env.users_arrivals[:] = 0

    cache_bitmap = np.zeros((
        user_env.n_videos, 
        user_env.n_layers,
        user_env.n_tiles, 
        user_env.n_gops
    ), dtype=np.int8)
    cache_bitmap[:, 0, :, :] = 1  # base layer cached for all videos

    all_requests = []
    for gop in range(steps_to_run):
        enh_layer_v0 = np.zeros(n_tiles, dtype=int)
        enh_layer_v1 = np.zeros(n_tiles, dtype=int)

        random_tile_indices = random.sample(range(n_tiles), 4)
        for idx in random_tile_indices:
            enh_layer_v0[idx] = 1

        random_tile_indices = random.sample(range(n_tiles), 4)
        for idx in random_tile_indices:
            enh_layer_v1[idx] = 1

        cache_bitmap[0, 1, :, gop] = enh_layer_v0  # video 0 enh layer
        cache_bitmap[1, 1, :, gop] = enh_layer_v1  # video 1 enh layer

        print(cache_bitmap[0, 1, :, gop])
        print(cache_bitmap[1, 1, :, gop])

        reqs = user_env.step(
            du_bitmaps=None,
            mec_bitmap=cache_bitmap
        )

        for r in reqs:
            validate_request_struct(r, n_tiles)
            enh_tiles = [t for t in r['tiles'] if t['layer'] == 1]
            print(f"  User {r['u']} Video {r['video']} GOP {r['gop']} EnhTiles={len(enh_tiles)}")

        all_requests.extend(reqs)

        # reqs = user_env.step(
        #     du_bitmaps=None,
        #     mec_bitmap=cache_bitmap
        # )

    print('\nSummary:')
    print(f'Total requests collected: {len(all_requests)}')
    base_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==0)
    enh_sizes = set(t['size'] for r in all_requests for t in r['tiles'] if t['layer']==1)
    print(f'Base layer size values: {base_sizes}')
    print(f'Enh layer size values: {enh_sizes}')

    print('\nSample request (first user, first step):')
    # Print all requests in a readable format
    print(json.dumps(all_requests, indent=4))

===== UserTileRequestEvents Test Harness =====
Grid: 4x3 -> 12 tiles | Users: 10 | Layers: 2
Running steps...
[0 0 0 1 1 1 1 0 0 0 0 0]
[1 0 0 0 1 0 1 0 0 0 1 0]
{'gop': 0, 'u': 0, 'p': 4, 'layer': 0, 'video': 53, 'tiles': []}
  User 0 Video 53 GOP 0 EnhTiles=0
{'gop': 0, 'u': 1, 'p': 4, 'layer': 0, 'video': 97, 'tiles': []}
  User 1 Video 97 GOP 0 EnhTiles=0
{'gop': 0, 'u': 2, 'p': 0, 'layer': 0, 'video': 18, 'tiles': []}
  User 2 Video 18 GOP 0 EnhTiles=0
{'gop': 0, 'u': 3, 'p': 0, 'layer': 0, 'video': 5, 'tiles': []}
  User 3 Video 5 GOP 0 EnhTiles=0
{'gop': 0, 'u': 4, 'p': 4, 'layer': 0, 'video': 18, 'tiles': []}
  User 4 Video 18 GOP 0 EnhTiles=0
{'gop': 0, 'u': 5, 'p': 3, 'layer': 0, 'video': 32, 'tiles': []}
  User 5 Video 32 GOP 0 EnhTiles=0
{'gop': 0, 'u': 6, 'p': 0, 'layer': 0, 'video': 77, 'tiles': []}
  User 6 Video 77 GOP 0 EnhTiles=0
{'gop': 0, 'u': 7, 'p': 1, 'layer': 0, 'video': 21, 'tiles': []}
  User 7 Video 21 GOP 0 EnhTiles=0
{'gop': 0, 'u': 8, 'p': 1, 'layer': 0, '